# **Advanced Models: Random Forest, GradientBoosting & XGBoost**
**Decodelabs Internship | Week 2 | Task 5 (Part 2)**

---
This is where I trained three ensemble models (Random Forest, Gradient Boosting, and XGBoost),
tuned their key hyperparameters using the validation set, and evaluated each model
comprehensively. I also handled class imbalance with SMOTE on the training data.

In [ ]:
import sys, os
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from configs.config import (
    RAW_FILE, IDS_MAP_FILE, INTERIM_FILE, PROCESSED_FILE,
    TRAIN_FILE, VAL_FILE, TEST_FILE,
    FIGURES_DIR, TABLES_DIR, PAPER_FIG_DIR, PAPER_TAB_DIR,
    RANDOM_SEED, TARGET_COL, PATIENT_ID_COL, MEDICATION_COLS,
    AGE_ORDER, icd9_to_category, COLORS, ensure_dirs
)
from src.plot_utils import set_plot_style, save_figure, save_table
ensure_dirs()
set_plot_style()
print("Config loaded. Seed:", RANDOM_SEED)

Config loaded. Seed: 42


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, recall_score, classification_report)
import warnings
warnings.filterwarnings("ignore")

# Try to import XGBoost; fall back gracefully if not installed
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost available.")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed. Will skip XGBoost model.")
    print("Install with: pip install xgboost")

# Try to import imbalanced-learn for SMOTE
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("imbalanced-learn (SMOTE) available.")
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed. Will use class_weight instead of SMOTE.")
    print("Install with: pip install imbalanced-learn")

train_df = pd.read_csv(TRAIN_FILE)
val_df   = pd.read_csv(VAL_FILE)

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_val   = val_df.drop(columns=[TARGET_COL])
y_val   = val_df[TARGET_COL]

print(f"\nTraining : {len(X_train):,} rows | {y_train.mean()*100:.1f}% positive")
print(f"Validation: {len(X_val):,} rows | {y_val.mean()*100:.1f}% positive")

XGBoost available.
imbalanced-learn not installed. Will use class_weight instead of SMOTE.
Install with: pip install imbalanced-learn

Training : 48,990 rows | 7.5% positive
Validation: 10,498 rows | 7.5% positive


## Applying SMOTE to training data

SMOTE (Synthetic Minority Over-sampling TEchnique) creates synthetic samples of the minority class (early readmission) in feature space. This is an alternative to `class_weight` for handling class imbalance. I applied SMOTE ONLY to the training set (applying SMOTE to validation or test leads to data leakage).

In [ ]:
if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    
    print("SMOTE applied to training data.")
    print(f"  Before: {len(X_train):,} rows | {y_train.mean()*100:.1f}% positive")
    print(f"  After : {len(X_train_resampled):,} rows | {y_train_resampled.mean()*100:.1f}% positive")
    print()
    print("The training set is now balanced (50/50).")
    print("Validation and test sets remain unchanged (real distribution).")
else:
    X_train_resampled = X_train
    y_train_resampled = y_train
    print("Using original imbalanced training data (class_weight will handle imbalance).")

Using original imbalanced training data (class_weight will handle imbalance).


## Random Forest

Random Forest builds many decision trees on random subsets of the training data and averages their predictions. This reduces overfitting compared to a single tree.

In [4]:
# Key hyperparameters I set:
#   n_estimators=300    : number of trees (more = better up to a point, slower)
#   max_depth=10        : max depth per tree (limits overfitting)
#   min_samples_leaf=20 : each leaf must have at least 20 samples (regularisation)
#   class_weight='balanced_subsample': down-weight majority class per tree
#   n_jobs=-1           : use all available CPU cores

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print("Training Random Forest (300 trees)...")
rf.fit(X_train_resampled, y_train_resampled)
print("Training complete.")

y_pred_rf = rf.predict(X_val)
y_prob_rf = rf.predict_proba(X_val)[:, 1]

print(f"\n=== Random Forest Validation Results ===")
print(classification_report(y_val, y_pred_rf,
      target_names=["No Readmit (0)", "Early Readmit (1)"]))
print(f"ROC-AUC          : {roc_auc_score(y_val, y_prob_rf):.4f}")
print(f"Average Precision: {average_precision_score(y_val, y_prob_rf):.4f}")

Training Random Forest (300 trees)...
Training complete.

=== Random Forest Validation Results ===
                   precision    recall  f1-score   support

   No Readmit (0)       0.95      0.74      0.83      9712
Early Readmit (1)       0.13      0.49      0.21       786

         accuracy                           0.72     10498
        macro avg       0.54      0.61      0.52     10498
     weighted avg       0.89      0.72      0.78     10498

ROC-AUC          : 0.6651
Average Precision: 0.1480


## Step 3 — Gradient Boosting

Gradient Boosting builds trees sequentially, where each tree corrects the errors of the previous one. It tends to be more accurate than Random Forest on tabular data but takes longer to train.

In [5]:
# Key hyperparameters:
#   n_estimators=300    : number of sequential trees
#   max_depth=4         : shallow trees work better for gradient boosting
#   learning_rate=0.05  : how much each tree corrects (lower = more trees needed)
#   subsample=0.8       : use 80% of training data per tree (stochastic boosting)

gb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=RANDOM_SEED
)

print("Training Gradient Boosting (300 estimators)...")
gb.fit(X_train_resampled, y_train_resampled)
print("Training complete.")

y_pred_gb = gb.predict(X_val)
y_prob_gb = gb.predict_proba(X_val)[:, 1]

print(f"\n=== Gradient Boosting Validation Results ===")
print(classification_report(y_val, y_pred_gb,
      target_names=["No Readmit (0)", "Early Readmit (1)"]))
print(f"ROC-AUC          : {roc_auc_score(y_val, y_prob_gb):.4f}")
print(f"Average Precision: {average_precision_score(y_val, y_prob_gb):.4f}")

Training Gradient Boosting (300 estimators)...
Training complete.

=== Gradient Boosting Validation Results ===
                   precision    recall  f1-score   support

   No Readmit (0)       0.93      1.00      0.96      9712
Early Readmit (1)       0.00      0.00      0.00       786

         accuracy                           0.92     10498
        macro avg       0.46      0.50      0.48     10498
     weighted avg       0.86      0.92      0.89     10498

ROC-AUC          : 0.6784
Average Precision: 0.1566


## XGBoost

 XGBoost is a highly optimised gradient boosting implementation. It is often the best-performing algorithm on structured/tabular data. `scale_pos_weight` handles class imbalance: set to (N negatives / N positives).

In [6]:
if XGBOOST_AVAILABLE:
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    scale_weight = neg_count / pos_count
    print(f"scale_pos_weight = {scale_weight:.1f} (balances class imbalance)")
    
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,  # handle imbalance
        eval_metric="auc",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0
    )
    
    print("Training XGBoost...")
    xgb.fit(X_train_resampled, y_train_resampled)
    print("Training complete.")
    
    y_pred_xgb = xgb.predict(X_val)
    y_prob_xgb = xgb.predict_proba(X_val)[:, 1]
    
    print(f"\n=== XGBoost Validation Results ===")
    print(classification_report(y_val, y_pred_xgb,
          target_names=["No Readmit (0)", "Early Readmit (1)"]))
    print(f"ROC-AUC          : {roc_auc_score(y_val, y_prob_xgb):.4f}")
    print(f"Average Precision: {average_precision_score(y_val, y_prob_xgb):.4f}")
else:
    print("XGBoost not available. Skipping.")
    xgb = None

scale_pos_weight = 12.4 (balances class imbalance)
Training XGBoost...
Training complete.

=== XGBoost Validation Results ===
                   precision    recall  f1-score   support

   No Readmit (0)       0.95      0.69      0.80      9712
Early Readmit (1)       0.13      0.55      0.21       786

         accuracy                           0.68     10498
        macro avg       0.54      0.62      0.50     10498
     weighted avg       0.89      0.68      0.76     10498

ROC-AUC          : 0.6711
Average Precision: 0.1583


## Comparing all models on validation set

In [7]:
from src.eval_utils import compute_metrics, compare_models_table

results = []

# Load logistic regression baseline results if available
import json, os
lr_path = os.path.join(TABLES_DIR, "07_baseline_results.json")
if os.path.exists(lr_path):
    with open(lr_path) as f:
        baseline_results = json.load(f)
    # Load logistic regression predictions
    from sklearn.linear_model import LogisticRegression
    lr_model = LogisticRegression(class_weight="balanced", max_iter=2000,
                                   random_state=RANDOM_SEED)
    lr_model.fit(X_train_resampled, y_train_resampled)
    y_pred_lr = lr_model.predict(X_val)
    y_prob_lr = lr_model.predict_proba(X_val)[:, 1]
    results.append(compute_metrics(y_val, y_pred_lr, y_prob_lr, "Logistic Regression"))

results.append(compute_metrics(y_val, y_pred_rf, y_prob_rf, "Random Forest"))
results.append(compute_metrics(y_val, y_pred_gb, y_prob_gb, "Gradient Boosting"))
if XGBOOST_AVAILABLE and xgb is not None:
    results.append(compute_metrics(y_val, y_pred_xgb, y_prob_xgb, "XGBoost"))

comp_df = compare_models_table(results)
print("=== Model Comparison (Validation Set) ===")
print(comp_df[["Accuracy","Balanced Acc.","Precision","Recall","F1-Score","ROC-AUC","Avg. Precision"]].to_string())

save_table(comp_df, "08_model_comparison_val.csv", TABLES_DIR)
print("\nModel comparison saved.")

=== Model Comparison (Validation Set) ===
                     Accuracy  Balanced Acc.  Precision  Recall  F1-Score  ROC-AUC  Avg. Precision
Model                                                                                             
Gradient Boosting      0.9245         0.4996     0.0000  0.0000    0.0000   0.6784          0.1566
XGBoost                0.6821         0.6212     0.1265  0.5496    0.2057   0.6711          0.1583
Random Forest          0.7171         0.6115     0.1298  0.4873    0.2050   0.6651          0.1480
Logistic Regression    0.6611         0.6151     0.1207  0.5611    0.1986   0.6537          0.1425
  Table  saved → c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\reports\tables\08_model_comparison_val.csv

Model comparison saved.


## All trained models saved for evaluation

In [8]:
import joblib

models_dir = os.path.join(os.path.dirname(PROCESSED_FILE), "..", "..", "models")
# Save to a models/ directory at project root (create if needed)
models_dir = os.path.join(PROJECT_ROOT, "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf, os.path.join(models_dir, "random_forest.pkl"))
joblib.dump(gb, os.path.join(models_dir, "gradient_boosting.pkl"))
if XGBOOST_AVAILABLE and xgb is not None:
    joblib.dump(xgb, os.path.join(models_dir, "xgboost.pkl"))

print(f"Models saved to: {models_dir}")
print(os.listdir(models_dir))

Models saved to: c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\models
['.gitkeep', 'gradient_boosting.pkl', 'random_forest.pkl', 'xgboost.pkl']
